# Companion Notebook — Chapter 09: AI Ethics, Bias & Responsible AI
**Artificial Intelligence Learning Hub** — *Institut Teknologi Sumatera (ITERA)*  
**Penulis**: Aji Pamungkas Tri Nurcahyo  

Notebook ini memandu Anda melakukan audit etika kecerdasan buatan secara komputasional:
1. **Audit Disparitas Historis**: Menyelidiki ketimpangan seleksi beasiswa masa lalu.
2. **Komputasi Metrik Keadilan Kelompok**: Demographic Parity, Equal Opportunity, dan Kaidah 80% EEOC.
3. **Mitigasi Pasca-Pemrosesan (*Post-processing Mitigation*)**: Penyesuaian ambang batas (*threshold tuning*) dan analisis *fairness-utility trade-off*.
4. **Audit Kerentanan Privasi Data**: Pembuktian serangan re-identifikasi (*linkage attack*) pada data terpseudonimisasi di bawah kerangka UU PDP (UU No. 27/2022).
5. **Matriks Penilaian Risiko Etis AI**: Evaluasi tingkat keparahan risiko berdasarkan kerangka kerja NIST AI RMF 1.0.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Membaca dataset historis pelamar beasiswa akademik
csv_path = Path("../code/data/scholarship_historical.csv")
df = pd.read_csv(csv_path)

print(f"Total Sampel Pelamar: {len(df)}")
print("\n5 Sampel Pertama:")
print(df.head())


Total Sampel Pelamar: 1000

5 Sampel Pertama:
  applicant_id demographic_group  test_score   gpa  extracurricular_score  true_potential  historical_decision
0     MHS-0001        Kelompok A       76.97  3.23                  72.06               1                    1
1     MHS-0002        Kelompok A       70.62  2.79                  86.37               0                    0
2     MHS-0003        Kelompok A       78.48  3.04                  68.10               1                    0
3     MHS-0004        Kelompok A       87.23  3.40                  78.18               1                    1
4     MHS-0005        Kelompok A       69.66  2.84                  79.99               0                    0


## 2. Investigasi Disparitas Historis vs Kemampuan Murni
Apakah tingkat kelulusan historis mencerminkan kemampuan objektif calon penerima beasiswa? Mari kita bandingkan rata-rata skor tes objektif dan proporsi kandidat berpotensi tinggi (*true potential*) antar kelompok.


In [2]:
summary = df.groupby('demographic_group').agg(
    n=('applicant_id', 'count'),
    mean_test=('test_score', 'mean'),
    mean_gpa=('gpa', 'mean'),
    true_potential_rate=('true_potential', 'mean'),
    historical_acceptance_rate=('historical_decision', 'mean')
).reset_index()

summary['disparate_impact_ratio'] = summary['historical_acceptance_rate'] / summary['historical_acceptance_rate'].iloc[0]
print(summary.to_string(index=False))


demographic_group   n  mean_test  mean_gpa  true_potential_rate  historical_acceptance_rate  disparate_impact_ratio
      Kelompok A 712  73.509747  3.064565             0.537921                    0.459270                1.000000
      Kelompok B 288  71.699931  2.990417             0.513889                    0.079861                0.173887


**Wawasan Analitis:**  
Potensi akademik objektif kedua kelompok sangat berimbang (~53.8% vs ~51.4%). Namun tingkat kelulusan historis menunjukkan ketimpangan ekstrim: 45.9% untuk Kelompok A vs **hanya 8.0% untuk Kelompok B**! Rasio *disparate impact* bernilai $0.1739$, melanggar aturan hukum 80% (*Four-Fifths Rule*).


## 3. Pelatihan Model Klasifikasi Standar & Komputasi Metrik Keadilan
Jika kita melatih model pembelajaran mesin tanpa kendala etika (*unconstrained model*) menggunakan label historis, model akan secara inheren menyerap dan memperkuat bias diskriminatif tersebut.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

X = df[['test_score', 'gpa', 'extracurricular_score']].values
y_hist = df['historical_decision'].values
y_true = df['true_potential'].values
groups = df['demographic_group'].values

# Split data 70:30
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.3, random_state=42, stratify=groups
)

model = LogisticRegression(random_state=42)
model.fit(X[train_idx], y_hist[train_idx])

# Evaluasi pada data uji
y_pred_test = model.predict(X[test_idx])
y_true_test = y_true[test_idx]
test_groups = groups[test_idx]

# Hitung TPR dan Selection Rate per kelompok
idx_A = (test_groups == "Kelompok A")
idx_B = (test_groups == "Kelompok B")

tpr_A = np.sum((y_pred_test[idx_A] == 1) & (y_true_test[idx_A] == 1)) / np.sum(y_true_test[idx_A] == 1)
tpr_B = np.sum((y_pred_test[idx_B] == 1) & (y_true_test[idx_B] == 1)) / np.sum(y_true_test[idx_B] == 1)

sr_A = np.mean(y_pred_test[idx_A])
sr_B = np.mean(y_pred_test[idx_B])

delta_eo = abs(tpr_A - tpr_B)
delta_dp = abs(sr_A - sr_B)

print(f"Akurasi Model Baseline: {accuracy_score(y_true_test, y_pred_test)*100:.2f}%")
print(f"Selection Rate A: {sr_A*100:.2f}% | Selection Rate B: {sr_B*100:.2f}% (Delta DP = {delta_dp:.4f})")
print(f"True Positive Rate (Recall) A: {tpr_A*100:.2f}%")
print(f"True Positive Rate (Recall) B: {tpr_B*100:.2f}%")
print(f"Kesenjangan Equal Opportunity (Delta EO): {delta_eo:.4f}")


Akurasi Model Baseline: 79.67%
Selection Rate A: 44.39% | Selection Rate B: 15.12% (Delta DP = 0.2928)
True Positive Rate (Recall) A: 78.26%
True Positive Rate (Recall) B: 29.55%
Kesenjangan Equal Opportunity (Delta EO): 0.4872


## 4. Mitigasi Pasca-Pemrosesan: Threshold Tuning & Trade-off
Untuk memperbaiki ketidakadilan tanpa harus mengulang pelatihan model dari nol, kita dapat menyesuaikan ambang batas keputusan (*classification threshold*) secara terpisah untuk Kelompok B.


In [4]:
probs_test = model.predict_proba(X[test_idx])[:, 1]

# Menurunkan ambang batas Kelompok B dari 0.50 ke 0.10
preds_mitigated = np.zeros_like(y_pred_test)
preds_mitigated[idx_A] = (probs_test[idx_A] >= 0.50).astype(int)
preds_mitigated[idx_B] = (probs_test[idx_B] >= 0.10).astype(int)

tpr_B_mit = np.sum((preds_mitigated[idx_B] == 1) & (y_true_test[idx_B] == 1)) / np.sum(y_true_test[idx_B] == 1)
acc_mit = accuracy_score(y_true_test, preds_mitigated)
delta_eo_mit = abs(tpr_A - tpr_B_mit)

print("=== HASIL PASCA-MITIGASI THRESHOLD ===")
print(f"Akurasi Global Pasca-Mitigasi: {acc_mit*100:.2f}%")
print(f"TPR Kelompok B Meningkat: {tpr_B*100:.2f}% -> {tpr_B_mit*100:.2f}%")
print(f"Delta EO Berkurang dari {delta_eo:.4f} -> {delta_eo_mit:.4f}")


=== HASIL PASCA-MITIGASI THRESHOLD ===
Akurasi Global Pasca-Mitigasi: 81.00%
TPR Kelompok B Meningkat: 29.55% -> 40.91%
Delta EO Berkurang dari 0.4872 -> 0.3735


## 5. Audit Privasi: Mengapa Masking Bukan Anonimisasi Sejati
Di bawah UU PDP No. 27/2022 dan GDPR, sekadar menutupi nama atau NIK (*pseudonymization*) tidak mencegah serangan re-identifikasi jika penyerang memiliki akses ke basis data eksternal (*quasi-identifier linkage attack*).


In [5]:
# Contoh Serangan Tautan Data Publik DPT
pseudonymized_record = pd.DataFrame([{
    'id_hash': '78aeaaa9d625', 'nama_masked': 'Ek********yo',
    'usia': 26, 'kelamin': 'L', 'kode_pos': '35145', 'diagnosis': 'Cardiac Arrhythmia'
}])

public_voter_registry = pd.DataFrame([{
    'nama_asli': 'Eko Prasetyo', 'usia': 26, 'kelamin': 'L', 'kode_pos': '35145',
    'alamat': 'Jl. Ryacudu No. 12, Sukarame'
}])

# Tautan pada Quasi-Identifiers
re_identified = pd.merge(pseudonymized_record, public_voter_registry, on=['usia', 'kelamin', 'kode_pos'])
print("Target Berhasil Di-reidentifikasi:")
print(f"Nama Asli: {re_identified['nama_asli'].iloc[0]}")
print(f"Diagnosis Rahasia: {re_identified['diagnosis'].iloc[0]}")
print(f"Alamat: {re_identified['alamat'].iloc[0]}")


Target Berhasil Di-reidentifikasi:
Nama Asli: Eko Prasetyo
Diagnosis Rahasia: Cardiac Arrhythmia
Alamat: Jl. Ryacudu No. 12, Sukarame


## 6. Ringkasan Prinsip Etika AI yang Bertanggung Jawab
1. **Fairness**: Wajib mengukur metrik keadilan kelompok formal, bukan hanya akurasi agregat.
2. **Accountability**: Sistem berisiko tinggi wajib mempertahankan pengawasan manusia (*Human-in-the-Loop*).
3. **Privacy by Design**: Terapkan $k$-anonymity dan batasi retensi data sesuai UU PDP No. 27/2022.
4. **Transparency**: Berikan penjelasan (*explanation*) atas keputusan otomatis kepada pihak terdampak.
